# files

> Files and cells over the gateway's contents and cells APIs

In [ ]:
#| default_exp files

[rustygate](https://github.com/AnswerDotAI/rustygate) serves two REST families beside the kernels API: a jupyter-inspired files API (`/api/contents`) and a cells API (`/api/cells`, per-cell operations on notebooks). This page builds their clients: `JupyAsyncFilesClient` addresses files and directories by path, `JupyAsyncCellsClient` binds to one notebook's cells, and `apply_ops` is the reference applier for the `cell_ops` change broadcasts that arrive on a kernel client's merged stream (`get_jmsg`, `channel` `'cells'`).

In [ ]:
#| export
import json
from base64 import b64encode, b64decode
from fastcore.basics import patch, patch_to
from fastspec.errors import APIError
from jupyasyncclient.core import KernelApi

In [ ]:
import asyncio, tempfile
from pathlib import Path
from queue import Empty
from fastcore.test import test_eq
from rustygate.tools import start_gateway
from jupyasyncclient import JupyAsyncKernelClient, JmsgQueues

## The files client

All three classes here share `KernelApi`'s HTTP plumbing: base URL, token, and the transport (a fresh client per request, or one you pass in). `session_id` is the author identity the gateway uses for echo suppression: a mutation carrying a kernel client's `session_id` is not broadcast back to that client's websocket. Left as `None`, writes carry no author and everyone subscribed hears them. A conditional write (`expected_hash=`) that loses the race raises `HashMismatch`, whose `hash` is the server's current file hash, which is what a retry needs.

In [ ]:
#| export
class HashMismatch(Exception):
    "A conditional write failed; `hash` is the server's current file hash."
    def __init__(self, hash):
        super().__init__(f'expected_hash is stale; the current hash is {hash}')
        self.hash = hash

class JupyAsyncFilesClient(KernelApi):
    "Files and directories over the gateway's contents API."
    def __init__(self, base_url, token=None, session_id=None, headers=None, timeout=30, http_client=None, verify=True):
        super().__init__(base_url, token=token, headers=headers, timeout=timeout, http_client=http_client, verify=verify)
        self.session_id = session_id

Every call goes through one helper. `_op` calls a spec-generated op with the author's `session_id` attached (ops without that parameter drop it), skips `None` values so optional query parameters stay absent, and converts a 409 carrying a hash into `HashMismatch`. `get`, `put`, `post`, and `patch` are its verb forms over the contents ops: the op splits `kwargs` into query parameters and body fields by the spec, so `get(path)` alone is the bare model and `put(path, type='directory')` is a whole request. `patch` routes `kwargs` through `body_`, because the rename body's `path` field shares its name with the route parameter.


In [ ]:
#| export
@patch
async def _op(self:JupyAsyncFilesClient, op, **kw):
    "Call spec op `op` with `session_id` attached and None values dropped; a conditional-write 409 raises `HashMismatch`."
    kw = {k:v for k,v in dict(kw, session_id=self.session_id).items() if v is not None}
    try: return await op(**kw)
    except APIError as e:
        if e.status_code==409 and isinstance(e.raw, dict) and 'hash' in e.raw: raise HashMismatch(e.raw['hash']) from e
        raise


`get`, `put`, and `post` map straight onto the contents endpoints:

In [ ]:
#| export
@patch
async def get(self:JupyAsyncFilesClient, path='', **kwargs):
    "The model at `path`; `kwargs` become query parameters, e.g. `fields`."
    if not path: return await self._op(self.api.contents.get_root, **kwargs)
    return await self._op(self.api.contents.get_path, path=path, **kwargs)

@patch
async def put(self:JupyAsyncFilesClient, path, expected_hash=None, **kwargs):
    "PUT to the contents API; `kwargs` are its body and query fields."
    return await self._op(self.api.contents.put_path, path=path, expected_hash=expected_hash, **kwargs)

@patch
async def post(self:JupyAsyncFilesClient, path, expected_hash=None, **kwargs):
    "POST to the contents API; `kwargs` are its body and query fields."
    return await self._op(self.api.contents.post_path, path=path, expected_hash=expected_hash, **kwargs)

`patch` frees the `path` name for the body, and `delete` removes a file or an empty directory:

In [ ]:
#| export
@patch_to(JupyAsyncFilesClient)
async def patch(self, path, /, expected_hash=None, **kwargs):
    "PATCH with `kwargs` as the JSON body; `path` is positional-only, freeing the name for the body."
    return await self._op(self.api.contents.patch_path, path=path, expected_hash=expected_hash, body_=kwargs)

@patch
async def delete(self:JupyAsyncFilesClient, path, expected_hash=None):
    "Delete a file or an empty directory."
    return await self._op(self.api.contents.delete_path, path=path, expected_hash=expected_hash)

In [ ]:
#| export
@patch
async def write(self:JupyAsyncFilesClient, path, content, expected_hash=None, unique=False):
    "Write `content` (`str` as text, `bytes` as base64), returning the model with its new `hash`; `unique` lands at a free `name_n.ext`."
    c,f = (b64encode(content).decode(),'base64') if isinstance(content, bytes) else (content,'text')
    return await self.put(path, expected_hash=expected_hash, unique=unique or None, content=c, format=f)

@patch
async def read(self:JupyAsyncFilesClient, path):
    "A file's contents: `str` for text, `bytes` for binary."
    m = await self.get(path, fields='content')
    return b64decode(m['content']) if m['format']=='base64' else m['content']

@patch
async def listing(self:JupyAsyncFilesClient, path='', fields=None):
    "The entries of directory `path`; `fields='hash'` adds each file's hash."
    return (await self.get(path, fields=fields))['content']


A live gateway to demonstrate against: the rustygate binary, serving a scratch directory as its files root.

In [ ]:
root = Path(tempfile.mkdtemp())
g = start_gateway(('rustygate', '--root', root))
fc = JupyAsyncFilesClient(g.url)
m = await fc.write('notes.txt', 'hello')
m

The same hash appears at every layer: the write's returned model, and the directory listing with `fields='hash'`.

In [ ]:
test_eq(await fc.read('notes.txt'), 'hello')
entry = next(e for e in await fc.listing(fields='hash') if e['name']=='notes.txt')
test_eq(entry['hash'], m['hash'])
await fc.get('notes.txt')

A stale `expected_hash` refuses the write and hands back the current hash. The retry needs no extra round trip:

In [ ]:
try: await fc.write('notes.txt', 'clobber', expected_hash='0'*64)
except HashMismatch as e: cur = e.hash
test_eq(cur, m['hash'])
m2 = await fc.write('notes.txt', 'hello2', expected_hash=cur)
assert m2['hash'] != cur
await fc.read('notes.txt')

`expected_hash=''` expects absence: the write must create. An existing path refuses with the usual `HashMismatch`:

In [ ]:
await fc.write('fresh.txt', 'made', expected_hash='')
try: await fc.write('fresh.txt', 'again', expected_hash='')
except HashMismatch as e: cur = e.hash
test_eq(await fc.read('fresh.txt'), 'made')
cur

Bytes round-trip through base64 without the caller seeing it:

In [ ]:
raw = bytes(range(256))
await fc.write('blob.bin', raw)
back = await fc.read('blob.bin')
test_eq(back, raw)
len(back)

`mkdir`, `rename`, and `copy` are one-line conveniences over the verbs:

In [ ]:
#| export
@patch
async def mkdir(self:JupyAsyncFilesClient, path, parents=False):
    "Create directory `path`; `parents` creates missing ancestors like `mkdir -p`."
    return await self.put(path, parents=parents or None, type='directory')

@patch
async def rename(self:JupyAsyncFilesClient, path, to):
    "Rename `path` to `to`, returning the new model."
    return await self.patch(path, path=to)

@patch
async def copy(self:JupyAsyncFilesClient, src, to, unique=False):
    "Copy `src` to `to`, returning the new model; `unique` lands at a free `name_n.ext`."
    return await self.post(to, unique=unique or None, copy_from=src)


In [ ]:
await fc.mkdir('sub')
await fc.mkdir('deep/a/b', parents=True)
await fc.copy('notes.txt', 'sub/notes.txt')
await fc.rename('sub/notes.txt', 'sub/renamed.txt')
test_eq([e['name'] for e in await fc.listing('sub')], ['renamed.txt'])
await fc.delete('sub/renamed.txt')
await fc.delete('sub')
await fc.delete('blob.bin')
[e['name'] for e in await fc.listing()]

`unique=True` never overwrites: the write or copy lands at the first free `name_n.ext`, and the returned model carries the final name:

In [ ]:
m3 = await fc.write('notes.txt', 'variant', unique=True)
test_eq(m3['name'], 'notes_1.txt')
m4 = await fc.copy('notes.txt', 'notes.txt', unique=True)
test_eq(m4['name'], 'notes_2.txt')
test_eq(await fc.read('notes_2.txt'), await fc.read('notes.txt'))
m4['path']

`edit` is the conditional read-modify-write: fetch content and hash, apply `f` to the parsed JSON, and put the result back guarded by `expected_hash`, repeating the whole read-transform-put when a concurrent writer wins the race. It is how a client changes something the cells API cannot address, such as a notebook's top-level metadata.

In [ ]:
#| export
@patch
async def edit(self:JupyAsyncFilesClient, path, f, tries=3):
    "Read the JSON file at `path`, apply `f` to the parsed object, and write it back conditionally, retrying a lost race"
    for _ in range(tries):
        m = await self.get(path, fields='content,hash')
        o = json.loads(m['content'])
        f(o)
        try: return await self.put(path, expected_hash=m['hash'], content=json.dumps(o, sort_keys=True, indent=1), format='text')
        except HashMismatch: pass
    raise RuntimeError(f'edit kept losing races for {path}')

In [ ]:
await fc.write('cfg.json', json.dumps(dict(a=1)))
def _bump(o): o['a'] += 1
await fc.edit('cfg.json', _bump)
test_eq(json.loads(await fc.read('cfg.json'))['a'], 2)

## The cells client

`JupyAsyncCellsClient` binds one notebook path in its constructor, and its cells methods need no path argument. The inherited file methods still take explicit paths, handy for a dialog's sibling assets. It keeps a `hash` cursor, updated by every response that carries the file hash: what a change broadcast's `hash` is compared against. `client[id]` awaits one cell; a tuple of ids awaits a list.

In [ ]:
#| export
class JupyAsyncCellsClient(JupyAsyncFilesClient):
    "One notebook's cells over the gateway's cells API."
    def __init__(self, base_url, path, token=None, session_id=None, headers=None, timeout=30, http_client=None, verify=True):
        super().__init__(base_url, token=token, session_id=session_id, headers=headers, timeout=timeout, http_client=http_client, verify=verify)
        self.path,self.hash,self.matched = str(path),None,None

    def __getitem__(self, ids): return self._lookup(ids)

In [ ]:
#| export
def _cs(v):
    "A comma-separated str from a str, an int, an iterable, or None"
    if v is None or isinstance(v, str): return v
    return ','.join(map(str, v)) if hasattr(v, '__iter__') else str(v)

@patch
async def cells(self:JupyAsyncCellsClient,
    ids=None, # Cell ids to keep: comma-separated str, or a list
    idx=None, # Cell positions to keep: comma-separated str, or a list of ints, 0-based, negative from the end; unions with `ids`
    q=None, # Keep only cells whose source matches this regex (multiline, smart-case)
    cell_type=None, # Keep only cells of this type: 'code', 'markdown', or 'raw'
    meta=None, # Keep only cells whose metadata contains this dict as a recursive subset; a None value means "key present"
    meta_not=None, # Drop cells whose metadata contains this dict as a recursive subset
    limit=None, # Keep at most this many cells after filtering
    context=None, # Also return this many neighbours either side of each kept cell; `self.matched` then names the true matches
    fields=None, # Comma-separated extras: 'hashes', 'meta', 'attachments'
):
    "The notebook's cells in document order, optionally selected and filtered (the gateway's cells GET stages)."
    m = await self._op(self.api.cells.get_cells, path=self.path, ids=_cs(ids), idx=_cs(idx), q=q, cell_type=cell_type,
        meta=None if meta is None else json.dumps(meta), meta_not=None if meta_not is None else json.dumps(meta_not),
        limit=limit, context=context, fields=fields)
    self.hash = m['hash']
    self.matched = m.get('matched')
    return m['cells']


`hashes` is the cheap form for sync, `apply` posts ops atomically, and `_lookup` resolves ids to cells:

In [ ]:
#| export
@patch
async def hashes(self:JupyAsyncCellsClient):
    "Per-cell `{'id','hash'}` rows: the cheap form for sync."
    m = await self._op(self.api.cells.get_cells, path=self.path, fields='hashes')
    self.hash = m['hash']
    return m['cells']

@patch
async def apply(self:JupyAsyncCellsClient, ops):
    "Apply `ops` atomically, returning ids of added cells."
    m = await self._op(self.api.cells.post_cells, path=self.path, ops=ops)
    self.hash = m['hash']
    return m['added_ids']

@patch
async def _lookup(self:JupyAsyncCellsClient, ids):
    one = isinstance(ids, str)
    want = [ids] if one else list(ids)
    got = await self.cells(ids=want)
    if len(got)!=len(want): raise KeyError(', '.join(i for i in want if i not in {c['id'] for c in got}))
    return got[0] if one else got

A notebook to work on, written through the files client (any nbformat producer works: the cells API reads the file fresh per request):

In [ ]:
cells = [dict(id='aaa1', cell_type='code', source='1+1', metadata={}, outputs=[], execution_count=None),
    dict(id='bbb2', cell_type='markdown', source='# hi', metadata={})]
await fc.write('d.ipynb', json.dumps(dict(nbformat=4, nbformat_minor=5, metadata={}, cells=cells)))
nb = JupyAsyncCellsClient(g.url, 'd.ipynb')
[c['id'] for c in await nb.cells()]

Cell lookup by id, fastlite-style; a missing id raises `KeyError`:

In [ ]:
c = await nb['bbb2']
test_eq(c['source'], '# hi')
pair = await nb['aaa1','bbb2']
test_eq([c['id'] for c in pair], ['aaa1','bbb2'])
try: await nb['nope']
except KeyError as e: err = str(e)
err

An op batch applies atomically: a sparse `add` is normalized server-side and its generated id comes back in order, an `update` replaces the named keys wholesale, and any failure rolls the whole batch back.

In [ ]:
added = await nb.apply([
    dict(op='add', cell=dict(cell_type='code', source='2+2'), after='aaa1'),
    dict(op='update', id='bbb2', source='# hello'),
])
new_id, = added
[c['id'] for c in await nb.cells()]

A 409 with no hash in its body is not a conditional-write race, and passes through as a plain `APIError`: the cells API answers 409 "cannot parse" for a file that is not a valid notebook, and a contents DELETE of a kernel-bound path answers 409 naming the remedy.

In [ ]:
await fc.write('junk.ipynb', '{not json')
bad = JupyAsyncCellsClient(g.url, 'junk.ipynb')
try: await bad.apply([dict(op='update', id='aaa1', source='x')])
except APIError as e: code = e.status_code
test_eq(code, 409)

## Applying ops

`apply_ops` is the client-side twin of the server's applier: the same vocabulary, applied to a plain list of cell dicts. It is how a client keeps a local view current from change broadcasts without refetching, and it bends rather than refuses, exactly as the server does. The one file-level op, `rename`, is the caller's business and raises here. A consumer that forgets to handle `rename` hears about it.

In [ ]:
#| export
def apply_ops(cells, ops):
    "Apply a `cell_ops` list to `cells` in place, in order, bending as the server does; returns `cells`"
    for o in ops:
        ids = [c['id'] for c in cells]
        op = o['op']
        if op=='update' and o['id'] in ids: cells[ids.index(o['id'])].update({k:v for k,v in o.items() if k not in ('op','id')})
        elif op in ('add','update'):
            c = dict(o['cell']) if op=='add' else {k:v for k,v in o.items() if k!='op'}
            if c.get('id') in ids: cells[ids.index(c['id'])].update({k:v for k,v in c.items() if k!='id'})
            else:
                c.setdefault('cell_type', 'code')
                c.setdefault('metadata', {})
                if c['cell_type']=='code':
                    c.setdefault('outputs', [])
                    c.setdefault('execution_count', None)
                anchor = o.get('after') or o.get('before')
                at = ids.index(anchor) + bool(o.get('after')) if anchor in ids else len(cells)
                cells.insert(at, c)
        elif op=='delete':
            if o['id'] in ids: del cells[ids.index(o['id'])]
        else: raise ValueError(f"unhandled op: {op}")
    return cells

Applying the ops we just sent to a stale local view reproduces the server's result exactly:

In [ ]:
view = await nb.cells()
ops = [dict(op='add', cell=dict(id='ccc3', cell_type='code', source='3', metadata={}, outputs=[], execution_count=None), before='aaa1'),
    dict(op='delete', id='bbb2')]
await nb.apply(ops)
apply_ops(view, ops)
test_eq([c['id'] for c in view], [c['id'] for c in await nb.cells()])
[c['id'] for c in view]

Ops never refuse, and the applier bends exactly as the server does: an update to a missing id materializes as an add at the end (normalized: `cell_type` defaults to code, code cells gain empty `outputs` and null `execution_count`), an add on an existing id updates that cell in place ignoring its anchor, a dead anchor appends, and a delete of a missing id is a no-op. The server's own application is the oracle here:

In [ ]:
ops = [dict(op='update', id='zzz9', source='9'), dict(op='delete', id='zzz8'),
    dict(op='add', cell=dict(id='ccc3', source='3b'), after='ggg7'), dict(op='add', cell=dict(id='ddd4', source='4'), after='ggg7')]
await nb.apply(ops)
apply_ops(view, ops)
srv = await nb.cells()
test_eq([c['id'] for c in view], [c['id'] for c in srv])
for i in ('zzz9', 'ccc3', 'ddd4'): test_eq(next(c for c in view if c['id']==i), next(c for c in srv if c['id']==i))


## Change broadcasts

A kernel created with a `path` binds to that notebook, and every client on its websocket hears about changes to it: `cell_ops` messages on the `cells` queue, content `{'path', 'hash', 'ops'}`. The author of a change hears nothing, where the author is whoever's `session_id` rode the mutation. `nb` above has no `session_id`, so its writes broadcast to everyone.

In [ ]:
kc = JupyAsyncKernelClient(g.url)
await kc.start_kernel(path='d.ipynb')
qs = JmsgQueues(kc, queues=('jmsg',), merge=dict(iopub='jmsg', stdin='jmsg', cells='jmsg'))
kc.start_channels()
await kc.wait_for_ready(timeout=60)
kc.channels_running

In [ ]:
await nb.apply([dict(op='update', id='aaa1', source='40+2')])
m = await qs.jmsg_for('cell_ops', timeout=15)
test_eq(m['header']['msg_type'], 'cell_ops')
m['content']

The broadcast keeps a local view current through `apply_ops`, and the message's `hash` says what the file must now hash to:

In [ ]:
apply_ops(view, m['content']['ops'])
test_eq(next(c['source'] for c in view if c['id']=='aaa1'), '40+2')
test_eq(m['content']['hash'], nb.hash)
m['content']['path']

Echo suppression, live: a cells client sharing the kernel client's `session_id` counts as the same author. Its writes produce no broadcast on `kc`'s queue:

In [ ]:
own = JupyAsyncCellsClient(g.url, 'd.ipynb', session_id=kc.session_id)
await own.apply([dict(op='update', id='aaa1', source='6*7')])
try:
    await qs.jmsg_for('cell_ops', timeout=1.5)
    heard = True
except Empty: heard = False
test_eq(heard, False)

Output persistence rides the same binding. An execute whose `metadata` carries `cellId` has its iopub output applied to that cell by the gateway, before the messages reach any client, so a fetch after the run's idle always shows them. An execute without `cellId`, or naming no cell in the file, is ephemeral and persists nothing:

In [ ]:
kc.execute('6*7', metadata=dict(cellId='aaa1'))
await qs.jmsg_for('status', pred=lambda m: m['content']['execution_state'] == 'idle', timeout=15)
c, = await nb.cells(ids=['aaa1'])
test_eq(c['outputs'][0]['data']['text/plain'], '42')
test_eq(c['execution_count'], 1)

Reads use the gateway's staged GET: `ids=` and `idx=` select cells (a union, in document order), `q=`, `cell_type=`, and `meta=`/`meta_not=` filter them, `limit=` caps the survivors, and `context=` adds neighbours, with `nb.matched` then naming the true matches. `meta=` takes a dict the cell's metadata must contain as a recursive subset, and a `None` value means "key present, whatever its value".

In [ ]:
test_eq([c['id'] for c in await nb.cells(idx=[0,-1])], ['ccc3', 'ddd4'])
test_eq([c['id'] for c in await nb.cells(ids='zzz9', idx=[0])], ['ccc3', 'zzz9'])
test_eq([c['id'] for c in await nb.cells(q=r'\*')], ['aaa1'])


`meta=` filters by recursive subset, a None value meaning "key present". In a merge op, a None value deletes its key, restoring the state:

In [ ]:
await nb.apply([dict(op='merge', id='zzz9', metadata=dict(tag=1))])
test_eq([c['id'] for c in await nb.cells(meta=dict(tag=None))], ['zzz9'])
await nb.apply([dict(op='merge', id='zzz9', metadata=dict(tag=None))])

`context=` returns neighbours around each hit, with `matched` naming the true matches:

In [ ]:
got = await nb.cells(q=r'\*', context=1)
test_eq((len(got), nb.matched), (3, ['aaa1']))
[c['id'] for c in got]

One file-level op can arrive in the stream: on `{'op':'rename','to':...}` update the client's `path`. Every broadcast op is self-sufficient: an add or update carries its whole cell, so applying one needs no prior state and no follow-up read. A bound notebook's op stream never resets, because the gateway holds the notebook as the current state: foreign writes with duplicate ids arrive repaired as ordinary ops, an unparseable foreign write changes nothing (reads keep serving the held state, and the gateway writes it back over the broken file after a grace interval), and a vanished bound file is restored the same way. After a reconnect that reports dropped messages, do not trust the stream: `hashes()` against the local view, then fetch the changed cells with `ids=`.


In [ ]:
#| hide
await kc.shutdown_kernel()
g.stop()

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()